# 데이터 결측치, 노이즈 처리
- 결측치 dropna 혹은 fillna(mean()) 등 사용 X
- 시계열 데이터에는 연속성 존재, 주변 데이터를 활용하는 방식 적용

In [10]:
import pandas as pd
import numpy as np

## 총 데이터 추출 횟수 정의
- 50 Run 가정
- 각 Run 당 100초간 공정 진행
- Total 데이터는 50*100 = 5000회의 Timestamp에서 측정

In [ ]:
np.random.seed(42)
runs = 50
seconds_per_run = 100 
total_rows = runs * seconds_per_run 


### TimeStamp 인덱스
- 측정된 TimeStamp 정의
- 26/08/01 10시 30분 00초부터 1초단위로 측정 (s)

In [ ]:
time_indices = pd.date_range(start="2026-08-01 10:30:00", periods=total_rows, freq="s") 
# 버전 문제 (s 소문자, 초단위 증가)


### 예시 데이터 생성
- Chamber 내 Pressure 데이터
- 기본 설정된 Base_Pressure : 150.0 mTorr
- 고주파 노이즈 (White Noise) : Gaussian 분포 (평균 0, 표준편차 3)
- Drift : 장기적인 Trend 변화 : 0~10으로 점차 증가

최종 Chamber 내 Prssure는 Base + Noise + Drift

In [ ]:
base_pressure = 150.0
noise = np.random.normal(loc=0, scale=3, size=total_rows) # 고주파 노이즈
drift = np.linspace(0, 10, total_rows) # 점진적 배기 밸브 오염에 의한 Drift 성분
pressure_values = base_pressure + noise + drift

### 결측치 시뮬레이션
- 설비 Idle 상황 가정
- 전체 5000개 Timestamp 중 1500~1600 구간 101개 결측치 발생
- 최종 Trace Data : Timestamp, Run ID, Step_No -> Pressure

In [ ]:
pressure_values[1500:1600] = np.nan

df_trace = pd.DataFrame({
    "Timestamp": time_indices,
    "Run_ID": np.repeat(np.arange(1, runs + 1), seconds_per_run),
    "Step_No": [1, 2, 3, 4, 5] * (total_rows // 5),
    "Pressure": pressure_values
})

## 데이터 정제 및 스무딩
### 정제 대상 설정
- Step_No : 3 (실제 식각 과정) 선택적 추출

In [ ]:
df_active = df_trace[df_trace["Step_No"] == 3].copy()

### 결측치 처리
- Dropna() : 삭제 -> 실제로는 X, 결측치 발생한 부분을 제거하면 문제 원인 분석에 불리
- fillna(mean()) : 평균으로 채우기 -> 주변 값들과 다른 양상 보일 수 있음

- 결론 : 결측치 복구를 위해 시계열 데이터 내에서 주변 값들에 대한 연속성을 고려
    - 선형 보간으로 해당 구간 밖의 데이터 활용한 처리
    - ffill 사용해 앞의 값을 붙여넣기 하는 방식도 가능 (Option)

In [ ]:
df_active["Pressure_Cleaned"] = df_active["Pressure"].interpolate(method="linear")
# df_active["Pressure_Cleaned"] = df_active["Pressure"].ffill()

### Rolling Window (시계열 이동평균) 활용 스무딩
- 이동평균을 통해 노이즈 완화
    - 각 시점에 자신 포함 앞의 window개 (20으로 설정) 데이터 mean 처리해서 값으로 적용
    - min_periods = 1 -> 데이터 초반부 20개 없어도 있는 값으로 평균 및 처리

In [ ]:
df_active["Pressure_Smoothed"] = df_active["Pressure_Cleaned"].rolling(window=20, min_periods=1).mean()

## 결과 분석
### 결측치 보정 결과 확인
- NaN 값이 있는지 처리 전과 후를 비교

In [9]:
# 결과 검증 출력
print("--- Check Missing Values Remained ---")
print(df_active[["Pressure", "Pressure_Cleaned"]].isna().sum())


--- Check Missing Values Remained ---
Pressure            20
Pressure_Cleaned     0
dtype: int64


### 인터랙티브 그래프 플롯
- Plotly 라이브러리
    - 확대/축소
    - Hover Tooltip : 값 확인 가능

In [ ]:
import plotly.express as px
fig = px.line(df_active, x='Timestamp', y=['Pressure', 'Pressure_Smoothed'], title="Chamber Pressure Analysis")
fig.show()